<a href="https://colab.research.google.com/github/SiJU-Ani/AnotherOneTest1/blob/main/Modular_NS-MABDO_Python_prototype_with_OOP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NSMABDO System

## Overview
The NSMABDO system simulates a multi-agent optimization and verification pipeline for compiler IR transformations.

---

## Components

### 1. Semantic Multi-Agent Profiler
Simulates:
- Selector → chooses passes
- Analyzer → analyzes IR
- Profiler → builds Semantic Intent Map

**Output:**
Semantic Intent Map (SIM)

---

### 2. VCWM (Verification-Centric Weight Model)

Evaluates optimization sequences and returns:
- Predicted IR state
- Reward vector → [r1, r2, r3, ...]
- Scalar reward

**Formula:**

R_scalar = Σ (wi * ri)


---

### 3. Bi-Directional Optimization Generative Engine (IIBO)

Behavior:
- Starts with intentionally bad sequence
- Improves using feedback
- Supports forward + backward optimization

**Update:**

Seq(t+1) = f(Seq(t), diagnostics)


---

### 4. Continuous Formal Verification Gate

Simulates SMT validation:
- Checks correctness
- Produces diagnostics if invalid

**Output:**

Valid = 0 or 1


---

### 5. NSMABDO Orchestrator

Controls loop:
- Runs multiple attempts
- Sends feedback to generator
- Stops when valid or max attempts reached

---

## Execution Flow

1. Input IR
2. Generate Semantic Intent Map
3. Generate initial (bad) sequence
4. Evaluate using VCWM
5. Verify using gate
6. If invalid:
   - Generate diagnostics
   - Update sequence
7. Repeat until valid

---

## Loop Representation


while Valid == 0:
SIM → Sequence → VCWM → Verify


---

## Dummy IR Example


define i32 @mock_function() {
ret i32 0
}


---

## Validation Result

- Attempt 1 → Invalid  
- Attempt 2 → Improved but invalid  
- Attempt 3 → Valid  

---

## Conclusion

System converges using:
- Feedback-driven optimization
- Semantic profiling
- Formal verification

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple
import random
import time


SemanticIntentMap = Dict[str, Any]
DiagnosticTrace = Dict[str, Any]
PassSequence = List[str]
RewardVector = Dict[str, float]




In [3]:
@dataclass
class IRState:
    """Represents a mock compiler IR state and metadata."""

    ir_code: str
    semantic_intent: SemanticIntentMap = field(default_factory=dict)
    history: List[str] = field(default_factory=list)


@dataclass
class WorldModelPrediction:
    """Prediction output from the Verification-Bound Compiler World Model."""

    predicted_state: IRState
    reward_vector: RewardVector
    scalar_reward: float


@dataclass
class VerificationResult:
    """Verification result from the Continuous Formal Verification Gate."""

    is_valid: bool
    diagnostic_trace: Optional[DiagnosticTrace] = None


class SemanticMultiAgentProfiler:
    """
    Simulates a selector/analyzer/profiler LLM agent network that builds
    a semantic intent map from raw LLVM-IR.
    """

    def __init__(self) -> None:
        self.selector_agent = "Selector"
        self.analyzer_agent = "Analyzer"
        self.profiler_agent = "Profiler"

    def profile(self, raw_ir: str, feedback_trace: Optional[DiagnosticTrace] = None) -> SemanticIntentMap:
        print("\n[Semantic Multi-Agent Profiler] Starting semantic profiling...")
        selected_regions = self._selector(raw_ir, feedback_trace)
        irregular_dfg = self._analyzer(raw_ir, selected_regions)
        intent_map = self._profiler(raw_ir, selected_regions, irregular_dfg, feedback_trace)
        print("[Semantic Multi-Agent Profiler] Semantic Intent Map built.")
        return intent_map

    def _selector(self, raw_ir: str, feedback_trace: Optional[DiagnosticTrace]) -> List[str]:
        print(f"  - {self.selector_agent}: selecting candidate hot regions...")
        base_regions = ["loop.header", "loop.body", "exit.block"]
        if feedback_trace:
            base_regions.append("repair.focus.block")
        return base_regions

    def _analyzer(self, raw_ir: str, selected_regions: Sequence[str]) -> Dict[str, str]:
        print(f"  - {self.analyzer_agent}: building irregular data-flow graph hints...")
        return {
            region: random.choice(["regular", "irregular", "phi-heavy"])
            for region in selected_regions
        }

    def _profiler(
        self,
        raw_ir: str,
        selected_regions: Sequence[str],
        irregular_dfg: Dict[str, str],
        feedback_trace: Optional[DiagnosticTrace],
    ) -> SemanticIntentMap:
        print(f"  - {self.profiler_agent}: producing semantic intent map...")

        hot_loops = [
            {
                "id": "L0",
                "trip_count_estimate": random.randint(64, 2048),
                "vectorization_candidate": random.choice([True, False]),
            }
        ]

        basic_block_partitions = {
            "cluster_A": ["entry", "loop.header"],
            "cluster_B": ["loop.body", "loop.latch"],
            "cluster_C": ["exit.block", "ret"],
        }

        correction_focus = (
            feedback_trace.get("suspect_passes", []) if feedback_trace else []
        )

        return {
            "raw_ir_digest": f"len={len(raw_ir)}",
            "selected_regions": list(selected_regions),
            "hot_loops": hot_loops,
            "irregular_data_flow_graph": irregular_dfg,
            "basic_block_partitions": basic_block_partitions,
            "correction_focus": correction_focus,
            "feedback_present": feedback_trace is not None,
        }




In [4]:
class VerificationBoundCompilerWorldModel:
    """
    Simulates a neural transition model (VCWM) that predicts rewards for a
    proposed pass trajectory.
    """

    def evaluate(self, current_state: IRState, pass_sequence: PassSequence) -> WorldModelPrediction:
        print("\n[VCWM] Simulating compiler transition and reward prediction...")

        deopt_penalty = self._deopt_penalty(pass_sequence)
        exploration_bonus = self._exploration_bonus(pass_sequence)

        cache_miss_prob = max(0.05, min(0.95, 0.45 + deopt_penalty * 0.15 - exploration_bonus * 0.1))
        vectorization_limit = max(0.10, min(0.99, 0.70 - deopt_penalty * 0.12 + exploration_bonus * 0.08))
        execution_latency = max(0.5, 1.3 + deopt_penalty * 0.25 - exploration_bonus * 0.30)

        reward_vector = {
            "cache_miss_probability": round(cache_miss_prob, 4),
            "vectorization_limit": round(vectorization_limit, 4),
            "execution_latency": round(execution_latency, 4),
        }

        # Higher scalar reward is better; lower cache miss and lower latency help.
        scalar_reward = (
            (1.0 - cache_miss_prob) * 0.40
            + vectorization_limit * 0.35
            + (1.0 / execution_latency) * 0.25
        )

        predicted_ir = (
            current_state.ir_code
            + f"\n; transformed_by={','.join(pass_sequence)}"
            + f"\n; predicted_reward={scalar_reward:.4f}"
        )
        predicted_state = IRState(
            ir_code=predicted_ir,
            semantic_intent=current_state.semantic_intent,
            history=current_state.history + ["VCWM-evaluated"],
        )

        print(f"[VCWM] Reward vector: {reward_vector}")
        print(f"[VCWM] Scalar reward: {scalar_reward:.4f}")

        return WorldModelPrediction(
            predicted_state=predicted_state,
            reward_vector=reward_vector,
            scalar_reward=round(scalar_reward, 4),
        )

    def _deopt_penalty(self, pass_sequence: PassSequence) -> float:
        deopt_markers = {
            "aggressive-loop-unroll",
            "scalarize-all",
            "disable-slp-vectorizer",
        }
        hits = sum(1 for p in pass_sequence if p in deopt_markers)
        return hits / max(1, len(pass_sequence))

    def _exploration_bonus(self, pass_sequence: PassSequence) -> float:
        if "aggressive-loop-unroll" in pass_sequence and "scalarize-all" in pass_sequence:
            # Bi-directional trajectory intentionally enters a de-optimizing region
            # before recovering with later optimization passes.
            return 0.45
        return 0.05





In [5]:
class BiDirectionalOptimizationGenerativeEngine:
    """
    Proposes pass sequences using IIBO:
    intentionally includes de-optimizing moves to escape local minima.
    """

    def generate(
        self,
        semantic_map: SemanticIntentMap,
        prior_failure_trace: Optional[DiagnosticTrace] = None,
        attempt: int = 1,
    ) -> PassSequence:
        print("\n[Bi-Directional Optimization Generative Engine] Generating pass sequence...")

        base_sequence: PassSequence = [
            "mem2reg",
            "gvn",
            "aggressive-loop-unroll",  # intentional de-opt candidate
            "scalarize-all",  # intentional de-opt candidate
            "loop-rotate",
            "slp-vectorizer",
            "instcombine",
        ]

        if prior_failure_trace:
            print("[Generator] Received diagnostic trace; adapting proposal...")
            suspect_passes = set(prior_failure_trace.get("suspect_passes", []))
            # Remove suspect passes and replace with safer alternatives.
            repaired_sequence = [p for p in base_sequence if p not in suspect_passes]
            repaired_sequence.extend(["loop-simplify", "licm"])  # correction tail
            # Keep order stable and unique for readability.
            deduped: PassSequence = []
            for p in repaired_sequence:
                if p not in deduped:
                    deduped.append(p)
            print(f"[Generator] Adapted sequence (attempt {attempt}): {deduped}")
            return deduped

        print(f"[Generator] Initial IIBO sequence (attempt {attempt}): {base_sequence}")
        return base_sequence




In [6]:
class ContinuousFormalVerificationGate:
    """
    Simulates an SMT-based verification gate (e.g., Alive2 style checks).
    """

    def __init__(self, invalid_probability: float = 0.45, seed: Optional[int] = None) -> None:
        self.invalid_probability = invalid_probability
        self.random = random.Random(seed)

    def verify(self, original_ir: str, pass_sequence: PassSequence) -> VerificationResult:
        print("\n[CFVG] Running formal equivalence verification...")

        # Bias toward invalid if sequence contains de-optimizing markers.
        deopt_heavy = {"aggressive-loop-unroll", "scalarize-all"}
        has_deopt = any(p in deopt_heavy for p in pass_sequence)
        invalid_threshold = self.invalid_probability + (0.20 if has_deopt else -0.10)
        invalid_threshold = min(max(invalid_threshold, 0.05), 0.95)

        if self.random.random() < invalid_threshold:
            trace = self._build_diagnostic_trace(pass_sequence)
            print("[CFVG] Result: INVALID trajectory (semantic hallucination risk).")
            print(f"[CFVG] Diagnostic trace: {trace}")
            return VerificationResult(is_valid=False, diagnostic_trace=trace)

        print("[CFVG] Result: VALID trajectory (semantic equivalence preserved).")
        return VerificationResult(is_valid=True, diagnostic_trace=None)

    def _build_diagnostic_trace(self, pass_sequence: PassSequence) -> DiagnosticTrace:
        suspects = [p for p in pass_sequence if p in {"aggressive-loop-unroll", "scalarize-all", "instcombine"}]
        if not suspects:
            suspects = [pass_sequence[-1]] if pass_sequence else ["unknown-pass"]

        return {
            "error": "non-equivalent transformation detected",
            "counterexample_input": "{x=13, y=2}",
            "suspect_passes": suspects,
            "hint": "reduce aggressive scalarization and enforce loop canonicalization",
        }




In [7]:
class NSMABDOSystem:
    """Coordinates all four NS-MABDO subsystems in a closed-loop optimization pipeline."""

    def __init__(
        self,
        profiler: SemanticMultiAgentProfiler,
        vcwm: VerificationBoundCompilerWorldModel,
        generator: BiDirectionalOptimizationGenerativeEngine,
        verifier: ContinuousFormalVerificationGate,
    ) -> None:
        self.profiler = profiler
        self.vcwm = vcwm
        self.generator = generator
        self.verifier = verifier

    def run(self, raw_ir: str, max_attempts: int = 8) -> Tuple[bool, int]:
        print("=" * 80)
        print("NS-MABDO Prototype: Neuro-Symbolic Multi-Agent Bi-Directional Optimizer")
        print("=" * 80)

        diagnostic_trace: Optional[DiagnosticTrace] = None
        semantic_map = self.profiler.profile(raw_ir=raw_ir, feedback_trace=diagnostic_trace)
        current_state = IRState(ir_code=raw_ir, semantic_intent=semantic_map)

        for attempt in range(1, max_attempts + 1):
            print(f"\n\n--- Optimization Attempt {attempt}/{max_attempts} ---")

            pass_sequence = self.generator.generate(
                semantic_map=current_state.semantic_intent,
                prior_failure_trace=diagnostic_trace,
                attempt=attempt,
            )

            prediction = self.vcwm.evaluate(current_state=current_state, pass_sequence=pass_sequence)
            verification = self.verifier.verify(original_ir=raw_ir, pass_sequence=pass_sequence)

            if verification.is_valid:
                print("\n[NS-MABDO] Accepted trajectory. Closed loop converged successfully.")
                print(f"[NS-MABDO] Final scalar reward: {prediction.scalar_reward:.4f}")
                print(f"[NS-MABDO] Final pass sequence: {pass_sequence}")
                return True, attempt

            # Autonomous self-correction loop.
            print("\n[NS-MABDO] Trajectory rejected. Feeding diagnostics back to profiler...")
            diagnostic_trace = verification.diagnostic_trace
            refreshed_map = self.profiler.profile(raw_ir=raw_ir, feedback_trace=diagnostic_trace)
            current_state = IRState(
                ir_code=raw_ir,
                semantic_intent=refreshed_map,
                history=current_state.history + ["CFVG-rejected"],
            )

            time.sleep(0.2)

        print("\n[NS-MABDO] Reached max attempts without a verified valid trajectory.")
        return False, max_attempts


def main() -> None:
    raw_ir = "define i32 @mock_function() { ... }"

    profiler = SemanticMultiAgentProfiler()
    vcwm = VerificationBoundCompilerWorldModel()
    generator = BiDirectionalOptimizationGenerativeEngine()
    verifier = ContinuousFormalVerificationGate(invalid_probability=0.40, seed=7)

    system = NSMABDOSystem(
        profiler=profiler,
        vcwm=vcwm,
        generator=generator,
        verifier=verifier,
    )

    success, attempts_used = system.run(raw_ir=raw_ir, max_attempts=6)

    print("\n" + "=" * 80)
    print("Run Summary")
    print("=" * 80)
    print(f"Verification success: {success}")
    print(f"Attempts used: {attempts_used}")



In [8]:

if __name__ == "__main__":
    main()

NS-MABDO Prototype: Neuro-Symbolic Multi-Agent Bi-Directional Optimizer

[Semantic Multi-Agent Profiler] Starting semantic profiling...
  - Selector: selecting candidate hot regions...
  - Analyzer: building irregular data-flow graph hints...
  - Profiler: producing semantic intent map...
[Semantic Multi-Agent Profiler] Semantic Intent Map built.


--- Optimization Attempt 1/6 ---

[Bi-Directional Optimization Generative Engine] Generating pass sequence...
[Generator] Initial IIBO sequence (attempt 1): ['mem2reg', 'gvn', 'aggressive-loop-unroll', 'scalarize-all', 'loop-rotate', 'slp-vectorizer', 'instcombine']

[VCWM] Simulating compiler transition and reward prediction...
[VCWM] Reward vector: {'cache_miss_probability': 0.4479, 'vectorization_limit': 0.7017, 'execution_latency': 1.2364}
[VCWM] Scalar reward: 0.6687

[CFVG] Running formal equivalence verification...
[CFVG] Result: INVALID trajectory (semantic hallucination risk).
[CFVG] Diagnostic trace: {'error': 'non-equivalent trans